In [ ]:
    import os
    import time
    import shutil
    import pathlib
    import itertools
    from PIL import Image

    import cv2
    import numpy as np
    import pandas as pd
    import seaborn as sns
    sns.set_style('darkgrid')
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import confusion_matrix, classification_report

    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.optimizers import Adam, Adamax
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
    from tensorflow.keras import regularizers

    import warnings
    warnings.filterwarnings("ignore")

    print ('modules loaded')

modules loaded


In [3]:
data_path = r"C:\Users\LENOVO\Desktop\sunflower\original" 

images = []
labels = []

for subfolder in os.listdir(data_path):
    
    subfolder_path = os.path.join(data_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue
  
    for image_filename in os.listdir(subfolder_path):
        image_path = os.path.join(subfolder_path, image_filename)
        images.append(image_path)
    
        labels.append(subfolder)
 
data = pd.DataFrame({'image': images, 'label': labels})


In [5]:
strat = data['label']
train_df, dummy_df = train_test_split(data,  train_size= 0.80, shuffle= True, random_state= 123, stratify= strat)

strat = dummy_df['label']
valid_df, test_df = train_test_split(dummy_df,  train_size= 0.5, shuffle= True, random_state= 123, stratify= strat)

In [6]:
print(train_df['label'].value_counts())
print(valid_df['label'].value_counts())
print(test_df['label'].value_counts())


label
Leaf scars      112
Fresh Leaf      107
Downy mildew     96
Gray mold        57
Name: count, dtype: int64
label
Fresh Leaf      14
Leaf scars      14
Downy mildew    12
Gray mold        7
Name: count, dtype: int64
label
Leaf scars      14
Fresh Leaf      13
Downy mildew    12
Gray mold        8
Name: count, dtype: int64


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# basic settings
img_size = 128
batch_size = 16
epochs = 15

# dataset path
dataset_path = r"C:\Users\LENOVO\Desktop\deiases pridection\original"

# data generator (train + validation)
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')   # 4 labels
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# training
model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs
)

# save model
model.save("image_classifier.h5")
print("✅ Model saved successfully")


Found 374 images belonging to 4 classes.
Found 92 images belonging to 4 classes.
Epoch 1/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 149s 6s/step - accuracy: 0.3877 - loss: 1.6546 - val_accuracy: 0.6196 - val_loss: 1.1106
Epoch 2/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 81s 3s/step - accuracy: 0.6283 - loss: 0.8897 - val_accuracy: 0.6522 - val_loss: 0.8508
Epoch 3/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 68s 3s/step - accuracy: 0.7406 - loss: 0.6652 - val_accuracy: 0.7174 - val_loss: 0.5441
Epoch 4/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 65s 3s/step - accuracy: 0.8316 - loss: 0.4639 - val_accuracy: 0.7935 - val_loss: 0.5622
Epoch 5/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 69s 3s/step - accuracy: 0.8743 - loss: 0.3765 - val_accuracy: 0.6848 - val_loss: 0.7963
Epoch 6/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 64s 3s/step - accuracy: 0.8904 - loss: 0.3108 - val_accuracy: 0.7609 - val_loss: 0.5764
Epoch 7/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 66s 3s/step - accuracy: 0.9332 - loss: 0.1879 - val_accuracy: 0.7391 - val_loss: 0.6823
Epoch 8/15
24/24 ━━━━━━━━━━━━━━━━━━━

✅ Model saved successfully


In [27]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# load trained model
model = tf.keras.models.load_model("image_classifier.h5")

# class names (IMPORTANT – auto from training)
class_names = list(model.class_names) if hasattr(model, "class_names") else [
    'Downy', 'Fresh', 'Gray', 'leaf scars'
]

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(128,128))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    print(f"🖼 Image: {img_path}")
    print(f"✅ Predicted Label: {predicted_class}")
    print(f"📊 Confidence: {confidence:.2f}%")

# example
predict_image(r"C:\Users\LENOVO\Desktop\deiases pridection\original\Gray mold\grayMold. (5).jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
🖼 Image: C:\Users\LENOVO\Desktop\deiases pridection\original\Gray mold\grayMold. (5).jpg
✅ Predicted Label: Gray
📊 Confidence: 99.71%
